In [1]:
import warnings
warnings.filterwarnings("ignore")
import random
import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
 
from sklearn.preprocessing import PowerTransformer

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D2 in response_OUS
data = list(OUS_D2['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D2, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

## Test dataset: MAASTRO 

In [6]:
(MAASTRO_D2['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [7]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [8]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D2['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [9]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D2, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]

In [10]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [11]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,shape_MinorAxisLength,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event


In [12]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [13]:
# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]
lower, upper = np.percentile(y_MAASTRO['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 374)
y_train:  (139,)


(99, 376)

# Feature Selection: RENT

In [14]:
selected_features = ["shape_Sphericity",
                     "glrlm_HighGrayLevelRunEmphasis_PET_c04",
                     "shape_MajorAxisLength",
                     "LBP_102_PET"]

# Selecting features in the DataFrame
X_rent = X[selected_features]

In [15]:
# Selecting features in the DataFrame
X_rent = X[selected_features]
X_new = X_rent.copy()

X_MAASTRO_rent = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_rent.copy()

# Yeo-Johnson Transformation

In [16]:
# Transform X_new 
# Set the categorical_columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Apply Yeo-Johnson transformation
pt = PowerTransformer(method='yeo-johnson')
X_new_numeric_transformed = pt.fit_transform(X_new_numeric)

# Create DataFrame with transformed numerical data
X_new_numeric_transformed = pd.DataFrame(X_new_numeric_transformed, 
                                         columns=X_new_numeric.columns, 
                                         index=X_new.index)

# Concatenate transformed numerical data with categorical data
X_new_std = pd.concat([X_new_numeric_transformed, X_new_categoric], axis=1)

# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the transformation for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = pt.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [17]:
X_new

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.761164,16.969770,42.073251,0.000000
1,0.697049,15.598394,24.613845,0.000000
2,0.565792,17.334294,48.030294,0.000034
3,0.684364,14.009277,25.589900,0.000000
4,0.503142,21.202180,34.684750,0.000199
...,...,...,...,...
134,0.742102,16.110383,33.069705,0.000000
135,0.722918,21.249575,41.043692,0.000000
136,0.652963,14.873884,36.618802,0.000000
137,0.724255,21.648860,45.870392,0.000000


In [18]:
X_new_std

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,1.124852,0.125487,0.072022,-0.784694
1,0.161153,-0.358143,-1.421627,-0.784694
2,-1.401990,0.250624,0.444135,0.236612
3,-0.012817,-0.946686,-1.314128,-0.784694
4,-1.982774,1.502519,-0.468644,1.784568
...,...,...,...,...
134,0.823106,-0.175124,-0.601731,-0.784694
135,0.532655,1.517088,0.002519,-0.784694
136,-0.421265,-0.622476,-0.316986,-0.784694
137,0.552472,1.639161,0.314722,-0.784694


In [19]:
MAASTRO_new

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.668072,19.034156,50.002093,0.000026
1,0.669961,11.392444,41.753334,0.000167
2,0.624081,14.567421,44.375483,0.000057
3,0.577624,13.477331,46.115989,0.000000
4,0.630933,16.365554,54.394967,0.000000
...,...,...,...,...
94,0.671754,17.492295,34.218615,0.000000
95,0.632189,14.267900,51.046869,0.000000
96,0.645548,12.143752,50.417953,0.000000
97,0.727488,15.300229,44.901412,0.000000


In [20]:
MAASTRO_new_std

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,-0.228627,0.816980,0.557388,0.033184
1,-0.204032,-1.995319,0.050605,1.682719
2,-0.770221,-0.736265,0.221607,0.693043
3,-1.281223,-1.151240,0.329735,-0.784694
4,-0.689674,-0.085023,0.794695,-0.784694
...,...,...,...,...
94,-0.180591,0.304443,-0.506427,-0.784694
95,-0.674760,-0.848664,0.615633,-0.784694
96,-0.513250,-1.682811,0.580714,-0.784694
97,0.600661,-0.466149,0.254712,-0.784694


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [21]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 01:09:42,635] A new study created in memory with name: no-name-f8ac055a-36cb-4324-ac86-71b1652da0a9


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.7767857142857143
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.7341772151898734


[I 2024-04-16 01:10:02,034] A new study created in memory with name: no-name-e7dce323-0925-4d30-a6ed-5b8463d3a1d5


Fold 5 C-index: 0.7089201877934272
[I 2024-04-16 01:10:02,017] Trial 0 finished with value: 0.7413282913937062 and parameters: {}. Best is trial 0 with value: 0.7413282913937062.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7413282913937062], datetime_start=datetime.datetime(2024, 4, 16, 1, 9, 42, 804098), datetime_complete=datetime.datetime(2024, 4, 16, 1, 10, 2, 15459), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7413282913937062


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.17729223997811025
Fold 2 IBS: 0.1563208701196685
Fold 3 IBS: 0.17242156247569246
Fold 4 IBS: 0.16843736811635485
Fold 5 IBS: 0.20228114907240044
[I 2024-04-16 01:10:03,104] Trial 0 finished with value: 0.17535063795244532 and parameters: {}. Best is trial 0 with value: 0.17535063795244532.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.17535063795244532], datetime_start=datetime.datetime(2024, 4, 16, 1, 10, 2, 83138), datetime_complete=datetime.datetime(2024, 4, 16, 1, 10, 3, 103816), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.17535063795244532


In [22]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [23]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.741
train_ibs:  0.175


#### Test

In [24]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [25]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.56
IBS score: 0.264


In [26]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [27]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [28]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 01:10:03,673] A new study created in memory with name: no-name-954dc7d1-d89f-44c7-82c0-b39f2cbef0b2


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.7745098039215687


[I 2024-04-16 01:10:04,499] A new study created in memory with name: no-name-d678922c-651e-4bcb-9f67-fa3871285227


Fold 4 C-index: 0.7067510548523207
Fold 5 C-index: 0.7230046948356808
[I 2024-04-16 01:10:04,470] Trial 0 finished with value: 0.7232935869123901 and parameters: {}. Best is trial 0 with value: 0.7232935869123901.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7232935869123901], datetime_start=datetime.datetime(2024, 4, 16, 1, 10, 3, 893094), datetime_complete=datetime.datetime(2024, 4, 16, 1, 10, 4, 469914), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7232935869123901


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397651518813013
Fold 2 IBS: 0.22157790765592797
Fold 3 IBS: 0.20453593880864498
Fold 4 IBS: 0.22473803370121878
Fold 5 IBS: 0.21812430704714597
[I 2024-04-16 01:10:05,250] Trial 0 finished with value: 0.2165905404802136 and parameters: {}. Best is trial 0 with value: 0.2165905404802136.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2165905404802136], datetime_start=datetime.datetime(2024, 4, 16, 1, 10, 4, 542141), datetime_complete=datetime.datetime(2024, 4, 16, 1, 10, 5, 250476), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2165905404802136


In [29]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [30]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.723
train_ibs:  0.217


#### Test

In [31]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [32]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.57


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [33]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [34]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 01:10:06,008] A new study created in memory with name: no-name-f27d6a5a-5a88-4df0-a516-5492148cce84


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.7341772151898734


[I 2024-04-16 01:10:07,036] A new study created in memory with name: no-name-5a67748c-8526-40b6-8c38-531483a0d833


Fold 5 C-index: 0.7089201877934272
[I 2024-04-16 01:10:07,028] Trial 0 finished with value: 0.7421336135225577 and parameters: {}. Best is trial 0 with value: 0.7421336135225577.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7421336135225577], datetime_start=datetime.datetime(2024, 4, 16, 1, 10, 6, 67854), datetime_complete=datetime.datetime(2024, 4, 16, 1, 10, 7, 27980), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7421336135225577


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.1779043098246738
Fold 2 IBS: 0.15504261131624741
Fold 3 IBS: 0.17188827223778808
Fold 4 IBS: 0.16854859140993816
Fold 5 IBS: 0.20132009844395674
[I 2024-04-16 01:10:07,952] Trial 0 finished with value: 0.17494077664652083 and parameters: {}. Best is trial 0 with value: 0.17494077664652083.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.17494077664652083], datetime_start=datetime.datetime(2024, 4, 16, 1, 10, 7, 69467), datetime_complete=datetime.datetime(2024, 4, 16, 1, 10, 7, 952319), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.17494077664652083


In [35]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [36]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.742
train_ibs:  0.175


#### Test 

In [37]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [38]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.561


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.262


In [39]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [40]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 01:10:08,559] A new study created in memory with name: no-name-6e0761dc-f1a3-4421-b82e-a7c61004cf6a


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.7341772151898734
Fold 5 C-index: 0.7089201877934272
[I 2024-04-16 01:10:09,518] Trial 0 finished with value: 0.7421336135225577 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7421336135225577.
Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.7089201877934272
[I 2024-04-16 01:10:10,466] Trial 1 finished with value: 0.7412897316660176 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.7421336135225577.
Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.7089201877934272
[I 2024-04-16 01:10:11,406] Trial 2 finished with value: 0.7412897316660176 and parameters: {'l1_ratio': 0.22692876841884668}. B

Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.7341772151898734
Fold 5 C-index: 0.7089201877934272
[I 2024-04-16 01:10:39,602] Trial 24 finished with value: 0.7421336135225577 and parameters: {'l1_ratio': 0.6776130739315861}. Best is trial 14 with value: 0.7421555325318184.
Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.7089201877934272
[I 2024-04-16 01:10:40,815] Trial 25 finished with value: 0.7412897316660176 and parameters: {'l1_ratio': 0.3339178579541805}. Best is trial 14 with value: 0.7421555325318184.
Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.7341772151898734
Fold 5 C-index: 0.7089201877934272
[I 2024-04-16 01:10:41,886] Trial 26 finished with value: 0.7421336135225577 and parameters: {'l1_ratio': 0.7685574200470692

Fold 1 C-index: 0.7012987012987013
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.7089201877934272
[I 2024-04-16 01:11:17,192] Trial 48 finished with value: 0.744001725554482 and parameters: {'l1_ratio': 0.04789647795907515}. Best is trial 34 with value: 0.744001725554482.
Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.7089201877934272
[I 2024-04-16 01:11:18,125] Trial 49 finished with value: 0.7412897316660176 and parameters: {'l1_ratio': 0.4477016529538363}. Best is trial 34 with value: 0.744001725554482.
Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.729957805907173
Fold 5 C-index: 0.7089201877934272
[I 2024-04-16 01:11:19,423] Trial 50 finished with value: 0.7412897316660176 and parameters: {'l1_ratio': 0.2311229033161103}. B

...

Fold 1 IBS: 0.17773835852772943
Fold 2 IBS: 0.1550120702715064
Fold 3 IBS: 0.17191349212398738
Fold 4 IBS: 0.1686465365477962
Fold 5 IBS: 0.20125460574969775
[I 2024-04-16 01:12:16,904] Trial 0 finished with value: 0.17491301264414344 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.17491301264414344.
Fold 1 IBS: 0.17740207527100574
Fold 2 IBS: 0.1550051342160191
Fold 3 IBS: 0.17206496653003014
Fold 4 IBS: 0.16887110007253892
Fold 5 IBS: 0.201283526483594
[I 2024-04-16 01:12:17,898] Trial 1 finished with value: 0.17492536051463756 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.17491301264414344.
Fold 1 IBS: 0.17733304969015584
Fold 2 IBS: 0.15508564163582447
Fold 3 IBS: 0.1721011248853764
Fold 4 IBS: 0.16891594462206458
Fold 5 IBS: 0.2013016264124419
[I 2024-04-16 01:12:18,955] Trial 2 finished with value: 0.17494747744917266 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.1749130126441434

Fold 2 IBS: 0.15500852885765679
Fold 3 IBS: 0.1719488571285054
Fold 4 IBS: 0.1686935937965669
Fold 5 IBS: 0.20128999680939944
[I 2024-04-16 01:12:39,690] Trial 25 finished with value: 0.1749314059256263 and parameters: {'l1_ratio': 0.6216865350760052}. Best is trial 17 with value: 0.17491286688139054.
Fold 1 IBS: 0.17783371963862415
Fold 2 IBS: 0.15503058028416897
Fold 3 IBS: 0.17188359234076203
Fold 4 IBS: 0.16857875987108376
Fold 5 IBS: 0.20128591032348828
[I 2024-04-16 01:12:40,445] Trial 26 finished with value: 0.17492251249162544 and parameters: {'l1_ratio': 0.9167426588025347}. Best is trial 17 with value: 0.17491286688139054.
Fold 1 IBS: 0.17780146768030844
Fold 2 IBS: 0.15502092537573706
Fold 3 IBS: 0.1719189655031944
Fold 4 IBS: 0.16863076980848457
Fold 5 IBS: 0.20129725442279459
[I 2024-04-16 01:12:41,203] Trial 27 finished with value: 0.17493387655810383 and parameters: {'l1_ratio': 0.7631358125417971}. Best is trial 17 with value: 0.17491286688139054.
Fold 1 IBS: 0.17772627

Fold 1 IBS: 0.1777363161349135
Fold 2 IBS: 0.15502350795079944
Fold 3 IBS: 0.17191281828569827
Fold 4 IBS: 0.1686507917871518
Fold 5 IBS: 0.2012493140163311
[I 2024-04-16 01:13:08,939] Trial 50 finished with value: 0.17491454963497882 and parameters: {'l1_ratio': 0.6890155968874181}. Best is trial 32 with value: 0.17491244637233597.
Fold 1 IBS: 0.17781901340404105
Fold 2 IBS: 0.15502262848620885
Fold 3 IBS: 0.17187878852776323
Fold 4 IBS: 0.16859523044538952
Fold 5 IBS: 0.20124787333406283
[I 2024-04-16 01:13:10,918] Trial 51 finished with value: 0.17491270683949306 and parameters: {'l1_ratio': 0.8387446189209664}. Best is trial 32 with value: 0.17491244637233597.
Fold 1 IBS: 0.17781491267112218
Fold 2 IBS: 0.1550322398267501
Fold 3 IBS: 0.1719233029529743
Fold 4 IBS: 0.16860347162500414
Fold 5 IBS: 0.20123764810107314
[I 2024-04-16 01:13:12,250] Trial 52 finished with value: 0.17492231503538475 and parameters: {'l1_ratio': 0.8199294640662467}. Best is trial 32 with value: 0.1749124463

Fold 5 IBS: 0.20127589565325266
[I 2024-04-16 01:13:47,063] Trial 74 finished with value: 0.17491843219604022 and parameters: {'l1_ratio': 0.8948721301308191}. Best is trial 32 with value: 0.17491244637233597.
Fold 1 IBS: 0.17790106930857522
Fold 2 IBS: 0.1550411009903398
Fold 3 IBS: 0.17188736052271278
Fold 4 IBS: 0.16855487673687006
Fold 5 IBS: 0.20131296424349196
[I 2024-04-16 01:13:49,158] Trial 75 finished with value: 0.17493947436039797 and parameters: {'l1_ratio': 0.9814493707942191}. Best is trial 32 with value: 0.17491244637233597.
Fold 1 IBS: 0.1778056319312043
Fold 2 IBS: 0.15502297389888584
Fold 3 IBS: 0.17192053439630495
Fold 4 IBS: 0.16862226951632467
Fold 5 IBS: 0.20130691818719307
[I 2024-04-16 01:13:51,407] Trial 76 finished with value: 0.17493566558598256 and parameters: {'l1_ratio': 0.7799741126138751}. Best is trial 32 with value: 0.17491244637233597.
Fold 1 IBS: 0.17729275937943842
Fold 2 IBS: 0.15508949346144105
Fold 3 IBS: 0.17211152226867815
Fold 4 IBS: 0.168958

Fold 1 IBS: 0.17781752431167402
Fold 2 IBS: 0.15503349816380735
Fold 3 IBS: 0.1718783230139951
Fold 4 IBS: 0.16859821851160697
Fold 5 IBS: 0.20124416510113766
[I 2024-04-16 01:14:23,686] Trial 99 finished with value: 0.1749143458204442 and parameters: {'l1_ratio': 0.8318252313874105}. Best is trial 95 with value: 0.17491209596981155.


* Best trial for IBS: 
 FrozenTrial(number=95, state=TrialState.COMPLETE, values=[0.17491209596981155], datetime_start=datetime.datetime(2024, 4, 16, 1, 14, 17, 395691), datetime_complete=datetime.datetime(2024, 4, 16, 1, 14, 18, 493279), params={'l1_ratio': 0.8326034206112033}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=95, value=None)


* Best Score for IBS: 
 0.17491209596981155


In [41]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [42]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.744
train_ibs:  0.175


#### Test

In [43]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [44]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.06342138075355325)

test_cindex : 0.56


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.8326034206112033)

test_ibs:  0.262


In [45]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [46]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 01:14:24,443] A new study created in memory with name: no-name-14a58355-cc6b-4d83-b696-f10415170781


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7467532467532467
Fold 2 C-index: 0.8125
Fold 3 C-index: 0.7769607843137255
Fold 4 C-index: 0.6814345991561181
Fold 5 C-index: 0.7699530516431925
[I 2024-04-16 01:14:37,578] Trial 0 finished with value: 0.7575203363732566 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7575203363732566.
Fold 1 C-index: 0.7445887445887446
Fold 2 C-index: 0.8169642857142857
Fold 3 C-index: 0.75
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.7699530516431925
[I 2024-04-16 01:14:48,821] Trial 1 finished with value: 0.750478431579118 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'max_features': 'sqrt', 'mi

Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.7991071428571429
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.7341772151898734
Fold 5 C-index: 0.7746478873239436
[I 2024-04-16 01:17:39,108] Trial 16 finished with value: 0.765813594477808 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 16, 'max_depth': 20, 'n_estimators': 5, 'oob_score': True, 'max_samples': 0.8484814588885312, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.08082385841318356, 'warm_start': True}. Best is trial 14 with value: 0.7907144637948861.
Fold 1 C-index: 0.7597402597402597
Fold 2 C-index: 0.8526785714285714
Fold 3 C-index: 0.7867647058823529
Fold 4 C-index: 0.7236286919831224
Fold 5 C-index: 0.8028169014084507
[I 2024-04-16 01:17:43,112] Trial 17 finished with value: 0.7851258260885514 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 15, 'min_samples_leaf': 15, 'max_depth': 17, 'n_estimators': 106, 'oob_score': True, 'max_samples': 0.98170723046990

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.8303571428571429
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.7658227848101266
Fold 5 C-index: 0.8356807511737089
[I 2024-04-16 01:18:47,494] Trial 31 finished with value: 0.7982641653072841 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 14, 'min_samples_leaf': 7, 'max_depth': 8, 'n_estimators': 75, 'oob_score': True, 'max_samples': 0.745146001885883, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.05174429879930956, 'warm_start': True}. Best is trial 31 with value: 0.7982641653072841.
Fold 1 C-index: 0.7575757575757576
Fold 2 C-index: 0.8348214285714286
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.7742616033755274
Fold 5 C-index: 0.8309859154929577
[I 2024-04-16 01:18:50,222] Trial 32 finished with value: 0.8032544311992128 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 14, 'min_samples_leaf': 7, 'max_depth': 9, 'n_estimators': 78, 'oob_score': True, 'max_samples': 0.7397628776532771, 'm

...

Fold 5 IBS: 0.17698987901902175
[I 2024-04-16 01:31:31,025] Trial 31 finished with value: 0.1756223060197624 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 17, 'min_samples_leaf': 13, 'max_depth': 1, 'n_estimators': 495, 'oob_score': False, 'max_samples': 0.6141886010044044, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.050889504115644586}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17921499719523448
Fold 2 IBS: 0.14457659998302025
Fold 3 IBS: 0.18892068480933402
Fold 4 IBS: 0.1881785321534129
Fold 5 IBS: 0.17366220512930025
[I 2024-04-16 01:31:50,191] Trial 32 finished with value: 0.17491060385406038 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 473, 'oob_score': False, 'max_samples': 0.7145937356077748, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.040680386261001684}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17839650482493216
Fold 2 IBS: 0.

Fold 1 IBS: 0.16980857091041326
Fold 2 IBS: 0.15719115142034723
Fold 3 IBS: 0.1767449129839921
Fold 4 IBS: 0.19510331227382113
Fold 5 IBS: 0.1831755952242518
[I 2024-04-16 01:36:42,518] Trial 47 finished with value: 0.17640470856256513 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 20, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 479, 'oob_score': False, 'max_samples': 0.8914383434724882, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.24824101632322468}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.19349637716238946
Fold 2 IBS: 0.18021514079410297
Fold 3 IBS: 0.1797766992061227
Fold 4 IBS: 0.20805913242700583
Fold 5 IBS: 0.19958771263889286
[I 2024-04-16 01:37:07,724] Trial 48 finished with value: 0.19222701244570276 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 14, 'min_samples_leaf': 6, 'max_depth': 3, 'n_estimators': 439, 'oob_score': False, 'max_samples': 0.9967788477130068, 'max_features': 'auto', 'min_weight_fraction_

Fold 5 IBS: 0.17601995444169838
[I 2024-04-16 01:41:35,266] Trial 62 finished with value: 0.1766496500814655 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 16, 'min_samples_leaf': 9, 'max_depth': 3, 'n_estimators': 298, 'oob_score': True, 'max_samples': 0.6969767031486694, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.11947149985769817}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17325965795999854
Fold 2 IBS: 0.16035062956465257
Fold 3 IBS: 0.17539882197990753
Fold 4 IBS: 0.193362443315604
Fold 5 IBS: 0.1796554283287958
[I 2024-04-16 01:42:00,389] Trial 63 finished with value: 0.17640539622979168 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 15, 'min_samples_leaf': 10, 'max_depth': 1, 'n_estimators': 311, 'oob_score': True, 'max_samples': 0.6304198495967799, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.16702154024396243}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17417602273955624
Fold 2 IBS: 0.1480464

Fold 1 IBS: 0.16818314645426713
Fold 2 IBS: 0.15369061634554518
Fold 3 IBS: 0.18213459050433092
Fold 4 IBS: 0.19123211966917278
Fold 5 IBS: 0.17979973767912885
[I 2024-04-16 01:48:18,581] Trial 78 finished with value: 0.17500804213048896 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 487, 'oob_score': False, 'max_samples': 0.4897248144615137, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.053996872945124036}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.1682237549262069
Fold 2 IBS: 0.15295334400832536
Fold 3 IBS: 0.18214608764478374
Fold 4 IBS: 0.19238806563941324
Fold 5 IBS: 0.18028732465861996
[I 2024-04-16 01:48:37,043] Trial 79 finished with value: 0.17519971537546983 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 11, 'max_depth': 11, 'n_estimators': 500, 'oob_score': False, 'max_samples': 0.49575398669591153, 'max_features': 'auto', 'min_weight_frac

Fold 5 IBS: 0.1747309238915691
[I 2024-04-16 01:51:37,150] Trial 93 finished with value: 0.1752048898255005 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 10, 'max_depth': 4, 'n_estimators': 437, 'oob_score': False, 'max_samples': 0.5630698614554802, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.0944985972942633}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17026943747099219
Fold 2 IBS: 0.15255352423829832
Fold 3 IBS: 0.18160069813968083
Fold 4 IBS: 0.19096533404342483
Fold 5 IBS: 0.1822030877543707
[I 2024-04-16 01:51:46,705] Trial 94 finished with value: 0.17551841632935336 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 2, 'n_estimators': 410, 'oob_score': False, 'max_samples': 0.5751544947591161, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.14409687866880386}. Best is trial 11 with value: 0.17074634120633253.
Fold 1 IBS: 0.17475332961241374
Fold 2 IBS: 0.1452

In [47]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [48]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.846
train_ibs:  0.171


#### Test

In [49]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [50]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=13, max_features=None, max_leaf_nodes=12,
                     max_samples=0.48042970450395145, min_samples_leaf=1,
                     min_samples_split=4,
                     min_weight_fraction_leaf=0.0025914663927405594,
                     n_estimators=289, random_state=123, warm_start=True)

test_cindex:  0.602


RandomSurvivalForest(max_depth=1, max_features='auto', max_leaf_nodes=18,
                     max_samples=0.7351810575897255, min_samples_leaf=11,
                     min_samples_split=2,
                     min_weight_fraction_leaf=0.008231293935378081,
                     n_estimators=2, random_state=123)

test_ibs:  0.233


In [51]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [52]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 01:52:42,519] A new study created in memory with name: no-name-1e3f9b66-c586-48a8-85d7-5af9e09c8328


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.70995670995671
Fold 2 C-index: 0.8348214285714286
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.7257383966244726
Fold 5 C-index: 0.755868544600939
[I 2024-04-16 01:52:44,538] Trial 0 finished with value: 0.763120153205612 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.763120153205612.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 01:52:49,980] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}. Be

Fold 4 C-index: 0.7046413502109705
Fold 5 C-index: 0.7136150234741784
[I 2024-04-16 01:53:54,861] Trial 15 finished with value: 0.7545769177214963 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 9, 'min_samples_leaf': 13, 'max_depth': 4, 'n_estimators': 258, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6880212408103346, 'min_weight_fraction_leaf': 0.07884420972256234}. Best is trial 12 with value: 0.7656344262692516.
Fold 1 C-index: 0.6645021645021645
Fold 2 C-index: 0.8191964285714286
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.7088607594936709
Fold 5 C-index: 0.676056338028169
[I 2024-04-16 01:53:58,548] Trial 16 finished with value: 0.7335270596877141 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is

Fold 1 C-index: 0.7186147186147186
Fold 2 C-index: 0.84375
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.7257383966244726
Fold 5 C-index: 0.7323943661971831
[I 2024-04-16 01:54:51,429] Trial 30 finished with value: 0.7619426335421767 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 12, 'min_samples_leaf': 1, 'max_depth': 9, 'n_estimators': 473, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.8382504072008882, 'min_weight_fraction_leaf': 0.09166005683798992}. Best is trial 19 with value: 0.7663281634231028.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.84375
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.7046413502109705
Fold 5 C-index: 0.7276995305164319
[I 2024-04-16 01:54:53,948] Trial 31 finished with value: 0.7601328692954678 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 16, 'min_samples_leaf': 15, 'max_depth': 3, 'n_estimators': 375, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_sa

...

Fold 1 IBS: 0.18923073618097216
Fold 2 IBS: 0.17632006586851645
Fold 3 IBS: 0.18707067058836963
Fold 4 IBS: 0.19307240934736852
Fold 5 IBS: 0.19343354994654252
[I 2024-04-16 02:01:49,683] Trial 30 finished with value: 0.18782548638635385 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 355, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.020774312173092817}. Best is trial 23 with value: 0.18218668005091396.
Fold 1 IBS: 0.1949036294792145
Fold 2 IBS: 0.1886371865011974
Fold 3 IBS: 0.19117863815442443
Fold 4 IBS: 0.2001198684681372
Fold 5 IBS: 0.19929856633981852
[I 2024-04-16 02:01:56,924] Trial 31 finished with value: 0.19482757778855841 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 2, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 396, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.

Fold 1 IBS: 0.18107213977138975
Fold 2 IBS: 0.17375539456626857
Fold 3 IBS: 0.18442566650093445
Fold 4 IBS: 0.19068424307356363
Fold 5 IBS: 0.18831741970696433
[I 2024-04-16 02:04:10,738] Trial 45 finished with value: 0.18365097272382416 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 11, 'min_samples_leaf': 2, 'max_depth': 17, 'n_estimators': 366, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.9992649320581628, 'min_weight_fraction_leaf': 0.15624439720920977}. Best is trial 44 with value: 0.17800229474539986.
Fold 1 IBS: 0.19953982633918327
Fold 2 IBS: 0.19967504623742535
Fold 3 IBS: 0.19245042867286613
Fold 4 IBS: 0.2085347403492104
Fold 5 IBS: 0.20542311562702958
[I 2024-04-16 02:04:21,336] Trial 46 finished with value: 0.20112463144514292 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 10, 'min_samples_leaf': 4, 'max_depth': 15, 'n_estimators': 471, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.8

Fold 1 IBS: 0.17659457791938485
Fold 2 IBS: 0.15948702371575116
Fold 3 IBS: 0.18725320428102982
Fold 4 IBS: 0.18157027728642194
Fold 5 IBS: 0.18138733472020935
[I 2024-04-16 02:06:15,406] Trial 60 finished with value: 0.17725848358455942 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 5, 'min_samples_leaf': 6, 'max_depth': 15, 'n_estimators': 346, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.7897702249078459, 'min_weight_fraction_leaf': 0.017004106952235656}. Best is trial 57 with value: 0.1764459724345519.
Fold 1 IBS: 0.17815851130122365
Fold 2 IBS: 0.15937759601308119
Fold 3 IBS: 0.18671233531081954
Fold 4 IBS: 0.18030514263303932
Fold 5 IBS: 0.18041925981213724
[I 2024-04-16 02:06:23,925] Trial 61 finished with value: 0.1769945690140602 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 5, 'min_samples_leaf': 6, 'max_depth': 15, 'n_estimators': 340, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.861

Fold 1 IBS: 0.21400073415496554
Fold 2 IBS: 0.2208296768873117
Fold 3 IBS: 0.20504439468604338
Fold 4 IBS: 0.22482948519570425
Fold 5 IBS: 0.21793226667188703
[I 2024-04-16 02:08:12,466] Trial 75 finished with value: 0.21652731151918242 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 6, 'min_samples_leaf': 7, 'max_depth': 11, 'n_estimators': 301, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.28939609623192764, 'min_weight_fraction_leaf': 0.38792353869222}. Best is trial 73 with value: 0.17587600438953827.
Fold 1 IBS: 0.18092605908791307
Fold 2 IBS: 0.1626714178491272
Fold 3 IBS: 0.1886398695722679
Fold 4 IBS: 0.18093264680250482
Fold 5 IBS: 0.18045471816506836
[I 2024-04-16 02:08:17,896] Trial 76 finished with value: 0.17872494229537628 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 4, 'min_samples_leaf': 5, 'max_depth': 14, 'n_estimators': 229, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.82756

Fold 1 IBS: 0.18417873567489768
Fold 2 IBS: 0.17215507074054423
Fold 3 IBS: 0.18759974758071274
Fold 4 IBS: 0.1889870504957803
Fold 5 IBS: 0.18950903071510605
[I 2024-04-16 02:09:59,065] Trial 90 finished with value: 0.1844859270414082 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 4, 'min_samples_leaf': 4, 'max_depth': 10, 'n_estimators': 262, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.9722689668575789, 'min_weight_fraction_leaf': 0.013698489831834436}. Best is trial 73 with value: 0.17587600438953827.
Fold 1 IBS: 0.17828049657703648
Fold 2 IBS: 0.15670442146325114
Fold 3 IBS: 0.1885595963781235
Fold 4 IBS: 0.17837564239035944
Fold 5 IBS: 0.17924664597888262
[I 2024-04-16 02:10:06,505] Trial 91 finished with value: 0.17623336055753064 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 5, 'max_depth': 15, 'n_estimators': 308, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.83

In [53]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [54]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.791
train_ibs:  0.176


#### Test

In [55]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [56]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=19, max_features=1, max_leaf_nodes=12,
                   max_samples=0.21404335777324493, min_samples_leaf=1,
                   min_samples_split=4,
                   min_weight_fraction_leaf=0.0013133682808468272,
                   n_estimators=405, random_state=123, warm_start=True)

C-index score: 0.572


ExtraSurvivalTrees(max_depth=14, max_features=None, max_leaf_nodes=6,
                   max_samples=0.8396800309305413, min_samples_leaf=5,
                   min_samples_split=5,
                   min_weight_fraction_leaf=0.02494499808293557,
                   n_estimators=305, oob_score=True, random_state=123)

IBS: 0.221


In [57]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [58]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 02:11:06,138] A new study created in memory with name: no-name-49783cdd-7073-4d27-a3fc-64fe44abf78a


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:11:43,761] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:12:03,034] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:22:07,874] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7610259439316011.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:23:23,548] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:36:31,346] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8840318412875596, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.27382248438555523, 'n_estimators': 385, 'criterion': 'squared_error', 'ccp_alpha': 2.0183033060186855, 'min_weight_fraction_leaf': 0.33643534713187806, 'max_features': 'auto', 'min_impurity_decrease': 5.889654690360788e-06, 'validation_fraction': 0.8062622793646869, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 9 with value: 0.7610259439316011.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:37:42,113] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7565765917190008, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.4300954216773497, 'n_estimators': 440, 'criterion': 'friedman

...

Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609557
[I 2024-04-16 04:24:14,709] Trial 44 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9992870370700113, 'learning_rate': 0.022847552015173876, 'dropout_rate': 0.1556807870961761, 'n_estimators': 451, 'criterion': 'squared_error', 'ccp_alpha': 1.6646055539220843, 'min_weight_fraction_leaf': 0.1403134453903068, 'max_features': 'auto', 'min_impurity_decrease': 2.7552659293421345e-07, 'validation_fraction': 0.8820166309308186, 'min_samples_split': 17, 'max_leaf_nodes': 14, 'min_samples_leaf': 10, 'max_depth': 2}. Best is trial 42 with value: 0.21286195228463267.
Fold 1 IBS: 0.21224782790603694
Fold 2 IBS: 0.2189260608342495
Fold 3 IBS: 0.20307124042767
Fold 4 IBS: 0.22313607826909793
Fold 5 IBS: 0.216439921823279
[I 2024-04-16 04:24:41,214] Trial 45 finished with value: 0.2147642258520667 and parameters: {'subsample': 0.8888212863898438, 'learning_rate': 0.0152498321107806

Fold 3 IBS: 0.20286393718880646
Fold 4 IBS: 0.22391249179729633
Fold 5 IBS: 0.21519882725417372
[I 2024-04-16 04:31:11,191] Trial 55 finished with value: 0.21426660724128474 and parameters: {'subsample': 0.9703353679292269, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.2676210325613897, 'n_estimators': 481, 'criterion': 'squared_error', 'ccp_alpha': 0.036860238643527846, 'min_weight_fraction_leaf': 0.21798842867076448, 'max_features': None, 'min_impurity_decrease': 4.086647023052284e-07, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 10, 'max_depth': 4}. Best is trial 42 with value: 0.21286195228463267.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-16 04:32:02,946] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.9556274374724505, 'learning_rate': 0.022777236

Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-16 04:34:55,036] Trial 67 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.922035492452401, 'learning_rate': 0.02138812894899817, 'dropout_rate': 0.2208120967013012, 'n_estimators': 121, 'criterion': 'squared_error', 'ccp_alpha': 0.29393333981111347, 'min_weight_fraction_leaf': 0.09999708192212713, 'max_features': None, 'min_impurity_decrease': 1.467292826198341e-07, 'validation_fraction': 0.8460725291615868, 'min_samples_split': 20, 'max_leaf_nodes': 14, 'min_samples_leaf': 10, 'max_depth': 12}. Best is trial 57 with value: 0.21250989041794505.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-16 04:34:56,852] Trial 68 finished with value: 0.21659054862241586 and parameters:

Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.2181243152560957
[I 2024-04-16 04:36:21,979] Trial 78 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9202091045129451, 'learning_rate': 0.019797687069268054, 'dropout_rate': 0.12670174450467558, 'n_estimators': 79, 'criterion': 'friedman_mse', 'ccp_alpha': 0.2910342652485194, 'min_weight_fraction_leaf': 0.15713001580184646, 'max_features': 'log2', 'min_impurity_decrease': 6.996091386072907e-06, 'validation_fraction': 0.7499357618644306, 'min_samples_split': 15, 'max_leaf_nodes': 15, 'min_samples_leaf': 15, 'max_depth': 16}. Best is trial 70 with value: 0.21173544073366624.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609582
[I 2024-04-16 04:36:23,295] Trial 79 finished with value: 0.21659054862241592 and parameters

Fold 2 IBS: 0.20607296701907554
Fold 3 IBS: 0.19811121797640907
Fold 4 IBS: 0.22011942798029224
Fold 5 IBS: 0.20776003972546755
[I 2024-04-16 04:37:11,639] Trial 89 finished with value: 0.2072255779812277 and parameters: {'subsample': 0.9081334064688743, 'learning_rate': 0.032246585462716706, 'dropout_rate': 0.10084377376256726, 'n_estimators': 92, 'criterion': 'friedman_mse', 'ccp_alpha': 0.019065157478907357, 'min_weight_fraction_leaf': 0.21843803937299006, 'max_features': None, 'min_impurity_decrease': 6.552978044824526e-05, 'validation_fraction': 0.8292287698601738, 'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 9, 'max_depth': 3}. Best is trial 89 with value: 0.2072255779812277.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609566
[I 2024-04-16 04:37:15,568] Trial 90 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8590448841763807, 

In [59]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [60]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.765
train_ibs:  0.207


#### Test

In [61]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [62]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.046986154574499193,
                                 dropout_rate=0.9141084507237711,
                                 learning_rate=0.005228819896792163,
                                 max_leaf_nodes=13,
                                 min_impurity_decrease=0.00017501601948817654,
                                 min_samples_leaf=15, min_samples_split=11,
                                 min_weight_fraction_leaf=0.2724947676380759,
                                 n_estimators=395, random_state=123,
                                 subsample=0.6128161811459047,
                                 validation_fraction=0.542037097833947)

C-index score: 0.579


GradientBoostingSurvivalAnalysis(ccp_alpha=0.019065157478907357,
                                 dropout_rate=0.10084377376256726,
                                 learning_rate=0.032246585462716706,
                                 max_leaf_nodes=15,
                                 min_impurity_decrease=6.552978044824526e-05,
                                 min_samples_leaf=9, min_samples_split=18,
                                 min_weight_fraction_leaf=0.21843803937299006,
                                 n_estimators=92, random_state=123,
                                 subsample=0.9081334064688743,
                                 validation_fraction=0.8292287698601738)

IBS: 0.218


In [63]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [64]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 04:37:57,028] A new study created in memory with name: no-name-9b2941b4-e901-4910-830a-76fe276abdef


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5584415584415584
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7352941176470589
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-16 04:37:58,395] Trial 0 finished with value: 0.6877154599339533 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6877154599339533.
Fold 1 C-index: 0.5584415584415584
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7352941176470589
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-16 04:38:07,005] Trial 1 finished with value: 0.6877154599339533 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6877154599339533.
Fold 1 C-index: 0.5800865800865801
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7401960784313726
Fold 4 C-index: 0.6624472573839663
Fold 5 C-ind

Fold 1 C-index: 0.6147186147186147
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7745098039215687
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.7370892018779343
[I 2024-04-16 04:39:18,671] Trial 19 finished with value: 0.7094407392934968 and parameters: {'subsample': 0.34314044045637715, 'dropout_rate': 0.34721836740782674, 'n_estimators': 248, 'learning_rate': 0.08042171957478578}. Best is trial 14 with value: 0.7199695842534164.
Fold 1 C-index: 0.6277056277056277
Fold 2 C-index: 0.7544642857142857
Fold 3 C-index: 0.7745098039215687
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.7370892018779343
[I 2024-04-16 04:39:19,629] Trial 20 finished with value: 0.7129309990337566 and parameters: {'subsample': 0.26117394341526556, 'dropout_rate': 0.18935257499862457, 'n_estimators': 112, 'learning_rate': 0.06551866052750378}. Best is trial 14 with value: 0.7199695842534164.
Fold 1 C-index: 0.6277056277056277
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.6877637

Fold 1 C-index: 0.6103896103896104
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7745098039215687
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7370892018779343
[I 2024-04-16 04:41:07,852] Trial 38 finished with value: 0.7068871747146159 and parameters: {'subsample': 0.5602770145716328, 'dropout_rate': 0.10259791290657247, 'n_estimators': 497, 'learning_rate': 0.045945647492717415}. Best is trial 24 with value: 0.72094756341247.
Fold 1 C-index: 0.6103896103896104
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7696078431372549
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.7323943661971831
[I 2024-04-16 04:41:16,281] Trial 39 finished with value: 0.7066555791346831 and parameters: {'subsample': 0.41312772203925596, 'dropout_rate': 0.23092838775300062, 'n_estimators': 464, 'learning_rate': 0.06709978541578802}. Best is trial 24 with value: 0.72094756341247.
Fold 1 C-index: 0.6320346320346321
Fold 2 C-index: 0.7544642857142857
Fold 3 C-index: 0.7745098039215687
Fold 4 C-index: 0.67510548523

...

Fold 3 IBS: 0.18031732971026723
Fold 4 IBS: 0.20229346331647807
Fold 5 IBS: 0.1878688249525523
[I 2024-04-16 04:48:29,183] Trial 19 finished with value: 0.19020375814222087 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 19 with value: 0.19020375814222087.
Fold 1 IBS: 0.19939419686684173
Fold 2 IBS: 0.18424688129359548
Fold 3 IBS: 0.18006097443217292
Fold 4 IBS: 0.2042629169306646
Fold 5 IBS: 0.18919912970114436
[I 2024-04-16 04:48:29,658] Trial 20 finished with value: 0.19143281984488383 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 19 with value: 0.19020375814222087.
Fold 1 IBS: 0.2020424213729644
Fold 2 IBS: 0.1900116559329881
Fold 3 IBS: 0.18492464173289203
Fold 4 IBS: 0.20602941149660942
Fold 5 IBS: 0.1924164075400163
[I 2024-04-16 04:48:29,986] Trial 21 finis

Fold 4 IBS: 0.22079633279247882
Fold 5 IBS: 0.18131480238918646
[I 2024-04-16 04:48:48,682] Trial 38 finished with value: 0.1916944913309321 and parameters: {'subsample': 0.10045702238474481, 'dropout_rate': 0.13272164755980653, 'n_estimators': 82, 'learning_rate': 0.0684395247111694}. Best is trial 19 with value: 0.19020375814222087.
Fold 1 IBS: 0.20327132380175603
Fold 2 IBS: 0.1860646272976219
Fold 3 IBS: 0.18261300691930407
Fold 4 IBS: 0.22057437433061436
Fold 5 IBS: 0.17996732471500895
[I 2024-04-16 04:48:49,642] Trial 39 finished with value: 0.19449813141286104 and parameters: {'subsample': 0.167666800643059, 'dropout_rate': 0.11471626893495929, 'n_estimators': 100, 'learning_rate': 0.050430345636818495}. Best is trial 19 with value: 0.19020375814222087.
Fold 1 IBS: 0.21673605085523345
Fold 2 IBS: 0.20773333805344843
Fold 3 IBS: 0.1943367194227906
Fold 4 IBS: 0.25172158643558845
Fold 5 IBS: 0.19362593239547257
[I 2024-04-16 04:48:50,322] Trial 40 finished with value: 0.2128307254

Fold 4 IBS: 0.21074415529612286
Fold 5 IBS: 0.1906046318704369
[I 2024-04-16 04:49:05,279] Trial 57 finished with value: 0.19362524680923224 and parameters: {'subsample': 0.28485371898030076, 'dropout_rate': 0.3370257678588191, 'n_estimators': 46, 'learning_rate': 0.04080281648872289}. Best is trial 41 with value: 0.18475802794319085.
Fold 1 IBS: 0.20640882920127823
Fold 2 IBS: 0.21970370169967193
Fold 3 IBS: 0.20347320840945904
Fold 4 IBS: 0.24194717585947562
Fold 5 IBS: 0.19629748402177308
[I 2024-04-16 04:49:06,492] Trial 58 finished with value: 0.21356607983833156 and parameters: {'subsample': 0.19606868118391554, 'dropout_rate': 0.4330958753331916, 'n_estimators': 132, 'learning_rate': 0.052258567546562856}. Best is trial 41 with value: 0.18475802794319085.
Fold 1 IBS: 0.2038886528943842
Fold 2 IBS: 0.1930918304872145
Fold 3 IBS: 0.18612714224929136
Fold 4 IBS: 0.20691981144697325
Fold 5 IBS: 0.19686372158553128
[I 2024-04-16 04:49:07,039] Trial 59 finished with value: 0.197378231

Fold 4 IBS: 0.2655628587347858
Fold 5 IBS: 0.24910597908314874
[I 2024-04-16 04:49:29,166] Trial 76 finished with value: 0.2583492427319435 and parameters: {'subsample': 0.1646209356841326, 'dropout_rate': 0.34829268282242476, 'n_estimators': 327, 'learning_rate': 0.049901380557131274}. Best is trial 41 with value: 0.18475802794319085.
Fold 1 IBS: 0.1998737329020388
Fold 2 IBS: 0.1821880269405455
Fold 3 IBS: 0.17751263839597953
Fold 4 IBS: 0.20328626032175104
Fold 5 IBS: 0.18402116294409254
[I 2024-04-16 04:49:29,842] Trial 77 finished with value: 0.18937636430088148 and parameters: {'subsample': 0.19315658624676152, 'dropout_rate': 0.41228050639999003, 'n_estimators': 61, 'learning_rate': 0.04804835804038757}. Best is trial 41 with value: 0.18475802794319085.
Fold 1 IBS: 0.20852668374519195
Fold 2 IBS: 0.23735501182444488
Fold 3 IBS: 0.21460136039744784
Fold 4 IBS: 0.25259421140638694
Fold 5 IBS: 0.19523514029924452
[I 2024-04-16 04:49:31,072] Trial 78 finished with value: 0.221662481

Fold 4 IBS: 0.22455751043206135
Fold 5 IBS: 0.1864269492522418
[I 2024-04-16 04:49:42,311] Trial 95 finished with value: 0.20061189013516972 and parameters: {'subsample': 0.7904534140423415, 'dropout_rate': 0.47309863500802474, 'n_estimators': 73, 'learning_rate': 0.04523657899468575}. Best is trial 41 with value: 0.18475802794319085.
Fold 1 IBS: 0.1960426079693441
Fold 2 IBS: 0.18418750895722966
Fold 3 IBS: 0.18357522731713158
Fold 4 IBS: 0.2056289060119681
Fold 5 IBS: 0.180314304167688
[I 2024-04-16 04:49:43,033] Trial 96 finished with value: 0.1899497108846723 and parameters: {'subsample': 0.1206876526650317, 'dropout_rate': 0.33764757365017084, 'n_estimators': 89, 'learning_rate': 0.05220949056295454}. Best is trial 41 with value: 0.18475802794319085.
Fold 1 IBS: 0.20530136961518963
Fold 2 IBS: 0.19194160718916256
Fold 3 IBS: 0.18472656896670164
Fold 4 IBS: 0.20871255663478866
Fold 5 IBS: 0.19409039803864542
[I 2024-04-16 04:49:43,344] Trial 97 finished with value: 0.19695450008889

In [65]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [66]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.723
train_ibs:  0.185


#### Test

In [67]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [68]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.1480636402691935,
                                              learning_rate=0.07459322757297186,
                                              n_estimators=322,
                                              random_state=123,
                                              subsample=0.12128259300543114)

C-index score: 0.565


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.24159494757459665,
                                              learning_rate=0.06615161558436967,
                                              n_estimators=56, random_state=123,
                                              subsample=0.10774190635914782)

IBS: 0.231


In [69]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [70]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.846,1.0
ExtraSurvivalTrees,0.791,2.0
GradientBoosting,0.765,3.0
CoxElastic,0.744,4.0
CoxLasso,0.742,5.0
CoxPH,0.741,6.0
CoxRidge,0.723,7.5
ComponentwiseGradientBoosting,0.723,7.5


In [71]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.171,1.0
CoxPH,0.175,3.0
CoxLasso,0.175,3.0
CoxElastic,0.175,3.0
ExtraSurvivalTrees,0.176,5.0
ComponentwiseGradientBoosting,0.185,6.0
GradientBoosting,0.207,7.0
CoxRidge,0.217,8.0


In [72]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
Randomsurvivalforest,0.602,1.0
GradientBoosting,0.579,2.0
ExtraSurvivalTrees,0.572,3.0
CoxRidge,0.570,4.0
ComponentwiseGradientBoosting,0.565,5.0
CoxLasso,0.561,6.0
CoxPH,0.560,7.5
CoxElastic,0.560,7.5


In [73]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
GradientBoosting,0.218,1.0
CoxRidge,0.221,2.5
ExtraSurvivalTrees,0.221,2.5
ComponentwiseGradientBoosting,0.231,4.0
Randomsurvivalforest,0.233,5.0
CoxLasso,0.262,6.5
CoxElastic,0.262,6.5
CoxPH,0.264,8.0


In [76]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d2/os/yeojohnson/rent/' 

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d2_os_yeojohnson_rent_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [77]:
from datetime import date

current_date = date.today()
print(current_date)

2024-04-16
